In [7]:
# Imports

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.tuner import Tuner
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Getting Data

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadmusharraf444/used-car-price-prediction-dataset")
data_path = os.path.join(path, "quikr_car.csv")

print("Path to dataset files:", data_path)

Path to dataset files: /home/franio/.cache/kagglehub/datasets/muhammadmusharraf444/used-car-price-prediction-dataset/versions/1/quikr_car.csv


In [36]:
data_df = pd.read_csv(data_path)

data_df.dropna(inplace=True)
data_df = data_df[data_df.Price != 'Ask For Price' ]
data_df.Price = data_df.Price.str.replace(',', '').astype(float)

data_df.kms_driven = data_df.kms_driven.str.strip('kms').str.replace(',', '', regex=False).str.strip().astype(float)
data_df.year = data_df.year.astype(int)
data_df = pd.get_dummies(data_df, columns=['company', 'fuel_type'], dtype=int)

data_df.drop(['name'], axis=1, inplace=True)

data_df

,year,Price,kms_driven,company_Audi,company_BMW,company_Chevrolet,company_Datsun,company_Fiat,company_Force,company_Ford,...,company_Nissan,company_Renault,company_Skoda,company_Tata,company_Toyota,company_Volkswagen,company_Volvo,fuel_type_Diesel,fuel_type_LPG,fuel_type_Petrol
0,2007,80000.0,45000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2006,425000.0,40.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,2014,325000.0,28000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,2014,575000.0,36000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
6,2012,175000.0,41000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2011,270000.0,50000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
885,2009,110000.0,30000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
886,2009,300000.0,132000.0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
888,2018,260000.0,27000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0


In [40]:
X,y = data_df.drop('Price', axis=1), data_df['Price']

X.dtypes

year                    int64
kms_driven            float64
company_Audi            int64
company_BMW             int64
company_Chevrolet       int64
company_Datsun          int64
company_Fiat            int64
company_Force           int64
company_Ford            int64
company_Hindustan       int64
company_Honda           int64
company_Hyundai         int64
company_Jaguar          int64
company_Jeep            int64
company_Land            int64
company_Mahindra        int64
company_Maruti          int64
company_Mercedes        int64
company_Mini            int64
company_Mitsubishi      int64
company_Nissan          int64
company_Renault         int64
company_Skoda           int64
company_Tata            int64
company_Toyota          int64
company_Volkswagen      int64
company_Volvo           int64
fuel_type_Diesel        int64
fuel_type_LPG           int64
fuel_type_Petrol        int64
dtype: object